# M2.1: Baseline Pipeline (57 raw features, LightGBM)

End-to-end pipeline: load LEAD data → downsampling → feature selection → CV split → StandardScaler → LightGBM → val AUC.

Resolves Unknown #2 (CV split building count) and Unknown #4 (downsampling class ratio).

## How to reproduce

```bash
cd ~/projects/lead-reproduction
git pull
uv sync
```

Then open this notebook in VS Code:
1. Top-right kernel selector → choose `.venv (Python 3.13.x)`
2. Restart Kernel and Run All
3. Expected: val AUC ≈ 0.895 (paper baseline: 0.9311, gap 3.86% < 5% pass)

## Load data & Anomaly rate inspection

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/train_features.csv")
print("Shape:", df.shape)
print("\ndtypes value_counts:")
print(df.dtypes.value_counts())
print("\nAnomaly rate:")
print(df["anomaly"].value_counts(normalize=True).rename("proportion"))
print("\nAnomaly absolute counts:")
print(df["anomaly"].value_counts())

Shape: (1749494, 57)

dtypes value_counts:
float64    35
int64      14
str         8
Name: count, dtype: int64

Anomaly rate:
anomaly
0    0.978682
1    0.021318
Name: proportion, dtype: float64

Anomaly absolute counts:
anomaly
0    1712198
1      37296
Name: count, dtype: int64


In [2]:
# Cloud coverage sentinel fix (buds-lab Feature generator Cells 4-5)
# 255 is the ASHRAE GEPIII missing/sentinel value encoding; replace with 10
sentinel_count = (df["cloud_coverage"] == 255).sum()
print(
    f"cloud_coverage = 255 count: {sentinel_count:,} ({sentinel_count / len(df) * 100:.1f}%)"
)
df["cloud_coverage"] = df["cloud_coverage"].replace({255: 10})
assert (df["cloud_coverage"] == 255).sum() == 0, "Sentinel values not fully replaced"
print("cloud_coverage 255 → 10 replacement verified")
print(
    f"New cloud_coverage stats: mean={df['cloud_coverage'].mean():.2f}, "
    f"min={df['cloud_coverage'].min()}, max={df['cloud_coverage'].max()}"
)

cloud_coverage = 255 count: 797,545 (45.6%)
cloud_coverage 255 → 10 replacement verified
New cloud_coverage stats: mean=5.24, min=0, max=10


## Downsampling

In [3]:
pos = df[df["anomaly"] == 1]
neg = df[df["anomaly"] == 0]
print(f"pos: {pos.shape[0]:,}  neg: {neg.shape[0]:,}")

negs1 = neg.sample(n=pos.shape[0], random_state=10)
negs2 = neg.sample(n=pos.shape[0], random_state=20)
df_eq = pd.concat([negs1, pos, negs2, pos], axis=0)

print(f"df_eq.shape: {df_eq.shape}")
print("\nClass ratio after downsampling:")
print(df_eq["anomaly"].value_counts(normalize=True))
print("\nClass counts:")
print(df_eq["anomaly"].value_counts())
# Unknown #4 resolved: 50:50, total 149184

pos: 37,296  neg: 1,712,198
df_eq.shape: (149184, 57)

Class ratio after downsampling:
anomaly
0    0.5
1    0.5
Name: proportion, dtype: float64

Class counts:
anomaly
0    74592
1    74592
Name: count, dtype: int64


## Feature selection

In [4]:
numeric_cols = df_eq.select_dtypes(
    include=["float64", "int64", "float32", "int32"]
).columns.tolist()
drop_cols = ["wind_direction", "air_temperature_std_lag73", "anomaly"]
feature_cols = [c for c in numeric_cols if c not in drop_cols]
print(f"Numeric cols before drop: {len(numeric_cols)}")
print(f"Feature count after drop: {len(feature_cols)}")  # expected 46

Numeric cols before drop: 49
Feature count after drop: 46


## CV split

In [5]:
train_df = df_eq[df_eq["building_id"] % 5 < 4]
val_df = df_eq[df_eq["building_id"] % 5 == 4]

print(
    f"Train: {train_df['building_id'].nunique()} buildings, {train_df.shape[0]:,} rows"
)
print(f"Val:   {val_df['building_id'].nunique()} buildings, {val_df.shape[0]:,} rows")
# Unknown #2 resolved: val=38 buildings (single holdout fold, not 5-fold loop)

Train: 162 buildings, 119,520 rows
Val:   38 buildings, 29,664 rows


## StandardScaler

In [6]:
from sklearn.preprocessing import StandardScaler

X_train = train_df[feature_cols].values
y_train = train_df["anomaly"].values
X_val = val_df[feature_cols].values
y_val = val_df["anomaly"].values

print(f"X_train NaN: {np.isnan(X_train).sum()}  X_val NaN: {np.isnan(X_val).sum()}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)

print(f"X_train_sc: {X_train_sc.shape}  X_val_sc: {X_val_sc.shape}")
print("StandardScaler OK (NaN preserved, LightGBM handles natively)")

X_train NaN: 3368  X_val NaN: 1254
X_train_sc: (119520, 46)  X_val_sc: (29664, 46)
StandardScaler OK (NaN preserved, LightGBM handles natively)


## LightGBM training

In [7]:
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import warnings

warnings.filterwarnings("ignore")

print("Training LGBMClassifier(n_estimators=100)...")
model = LGBMClassifier(n_estimators=100, verbose=-1)
model.fit(X_train_sc, y_train)

val_pred = model.predict_proba(X_val_sc)[:, 1]
val_auc = roc_auc_score(y_val, val_pred)
print(f"Val AUC: {val_auc:.6f}")
print(
    f"Paper baseline: 0.9311  |  Gap: {(0.9311 - val_auc) / 0.9311 * 100:.2f}%  (pass if < 5%)"
)

Training LGBMClassifier(n_estimators=100)...


Val AUC: 0.895207
Paper baseline: 0.9311  |  Gap: 3.85%  (pass if < 5%)


## Validation AUC summary

In [8]:
anomaly_rate = df["anomaly"].mean()
pos_count = int(df["anomaly"].sum())
neg_count = int((df["anomaly"] == 0).sum())
eq_class_ratio_normal = df_eq["anomaly"].value_counts(normalize=True)[0]
eq_class_ratio_anomaly = df_eq["anomaly"].value_counts(normalize=True)[1]

print("=" * 60)
print("M2.1 Done-when Summary")
print("=" * 60)
summary = f"""
| Metric                         | Value                    | Pass? |
|--------------------------------|--------------------------|-------|
| Raw CSV rows × cols            | {df.shape[0]:,} × {df.shape[1]}           | ✓     |
| Anomaly rate (actual)          | {anomaly_rate:.4%}              | ✓     |
| Pos count                      | {pos_count:,}                | ✓     |
| Neg count                      | {neg_count:,}            | ✓     |
| df_eq rows after downsampling  | {df_eq.shape[0]:,}               | ✓     |
| Downsampling class ratio (0:1) | {eq_class_ratio_normal:.1%}:{eq_class_ratio_anomaly:.1%}               | ✓     |
| Feature count after selection  | {len(feature_cols)}                       | ✓ (expected 46) |
| Train buildings                | {train_df["building_id"].nunique()}                      | ✓     |
| Val buildings                  | {val_df["building_id"].nunique()}                       | ✓     |
| Train rows                     | {train_df.shape[0]:,}               | ✓     |
| Val rows                       | {val_df.shape[0]:,}                | ✓     |
| StandardScaler pre-NaN (train) | {int(np.isnan(X_train).sum()):,}                | ✓     |
| LightGBM val AUC               | {val_auc:.4f}                | ✓ (paper: 0.9311, gap: {(0.9311 - val_auc) / 0.9311 * 100:.1f}% < 5%) |
"""
print(summary)
print("Unknown #2 resolved: val=38 buildings (single holdout fold)")
print("Unknown #4 resolved: downsampling class ratio 50:50, total rows 149,184")

M2.1 Done-when Summary

| Metric                         | Value                    | Pass? |
|--------------------------------|--------------------------|-------|
| Raw CSV rows × cols            | 1,749,494 × 57           | ✓     |
| Anomaly rate (actual)          | 2.1318%              | ✓     |
| Pos count                      | 37,296                | ✓     |
| Neg count                      | 1,712,198            | ✓     |
| df_eq rows after downsampling  | 149,184               | ✓     |
| Downsampling class ratio (0:1) | 50.0%:50.0%               | ✓     |
| Feature count after selection  | 46                       | ✓ (expected 46) |
| Train buildings                | 162                      | ✓     |
| Val buildings                  | 38                       | ✓     |
| Train rows                     | 119,520               | ✓     |
| Val rows                       | 29,664                | ✓     |
| StandardScaler pre-NaN (train) | 3,368                | ✓     |
| LightGB

In [9]:
old_auc = 0.8952  # M2.1 original (commit fefde05, no cloud_coverage fix)
delta_auc = val_auc - old_auc
print(f"\n{'=' * 60}")
print("M2.2.0 ΔAUC analysis")
print(f"{'=' * 60}")
print(f"M2.1 original AUC (no cloud_coverage fix): {old_auc:.4f}")
print(f"M2.2.0 new AUC (with fix):                 {val_auc:.4f}")
print(f"ΔAUC = +{delta_auc:.4f}")
print("Paper baseline: 0.9311")
print(f"Remaining gap:  {(0.9311 - val_auc) * 100:.2f}%")


M2.2.0 ΔAUC analysis
M2.1 original AUC (no cloud_coverage fix): 0.8952
M2.2.0 new AUC (with fix):                 0.8952
ΔAUC = +0.0000
Paper baseline: 0.9311
Remaining gap:  3.59%
